# Лаборатори 3: CSP хэрэгжүүлэлт

## Алхам 1: CSP класс

In [ ]:
import time

class TooSlow(Exception):
    pass

class CSP:
    """CSP: хувьсагч, домэйн, хөрш, хязгаарлалт, зангилааны тоолуур хадгална."""
    def __init__(self, variables, domains, neighbors, constraint):
        self.variables = variables
        self.domains = domains
        self.neighbors = neighbors
        self.constraint = constraint
        self.nodes = 0
        self.limit = None

    def is_consistent(self, var, value, assignment):
        """var=value оноовол оноогдсон хөршүүдтэй зөрчилдөх эсэхийг шалгана."""
        for n in self.neighbors[var]:
            if n in assignment and not self.constraint(var, value, n, assignment[n]):
                return False
        return True

    def unassigned(self, assignment):
        """Оноолтод ороогүй хувьсагчдын жагсаалт."""
        return [v for v in self.variables if v not in assignment]

def build_neighbors(variables, edges):
    """Ирмэг бүрийг хоёр чиглэлд нэмж neighbors dictionary үүсгэнэ."""
    nb = {v: [] for v in variables}
    for a, b in edges:
        nb[a].append(b); nb[b].append(a)
    return nb

**Шалгах:** Гараар 2–3 хувьсагчтай жижиг CSP үүсгээд is_consistent-ийг нэг
зөрчилтэй, нэг зөрчилгүй тохиолдолд туршина. Хоёр тохиолдолд хоёуланд нь
хүлээгдсэн хариу гарсан эсэхийг харна.

In [2]:
# Шалгах: хувьсагч A, B, C; хязгаарлалт: хөрш хоёр ижил утгатай байж болохгүй
nb = build_neighbors(["A","B","C"], [("A","B"),("B","C")])
t = CSP(["A","B","C"], {v:[1,2] for v in "ABC"}, nb, lambda X,x,Y,y: x != y)
print("B=1 (A=1 оноогдсон) ->", t.is_consistent("B", 1, {"A":1}), "(False хүлээгдэнэ)")
print("B=2 (A=1 оноогдсон) ->", t.is_consistent("B", 2, {"A":1}), "(True хүлээгдэнэ)")
print("C=1 (A=1 оноогдсон, C нь A-н хөрш биш) ->", t.is_consistent("C", 1, {"A":1}), "(True хүлээгдэнэ)")
print("unassigned:", t.unassigned({"A":1}))

B=1 (A=1 оноогдсон) -> False (False хүлээгдэнэ)
B=2 (A=1 оноогдсон) -> True (True хүлээгдэнэ)
C=1 (A=1 оноогдсон, C нь A-н хөрш биш) -> True (True хүлээгдэнэ)
unassigned: ['B', 'C']


## Алхам 2: Газрын зураг будах бодлого

In [3]:
def make_map_csp(colors=("red", "green", "blue")):
    """Австралийн газрын зургийг CSP хэлбэрээр буцаана."""
    variables = ["WA", "NT", "SA", "Q", "NSW", "V", "T"]
    domains = {v: list(colors) for v in variables}
    edges = [("WA","NT"),("WA","SA"),("NT","SA"),("NT","Q"),("SA","Q"),
             ("SA","NSW"),("SA","V"),("Q","NSW"),("NSW","V")]
    return CSP(variables, domains, build_neighbors(variables, edges), lambda A,a,B,b: a != b)

**Шалгах:** SA-ийн хөршийн тоо 5, T-ийнх 0, WA-ийнх 2 байна.

In [4]:
m = make_map_csp()
print({v: len(m.neighbors[v]) for v in m.variables})
assert len(m.neighbors["SA"]) == 5 and len(m.neighbors["T"]) == 0 and len(m.neighbors["WA"]) == 2
print("Шалгалт тэнцлээ: SA=5, T=0, WA=2")

{'WA': 2, 'NT': 3, 'SA': 5, 'Q': 3, 'NSW': 3, 'V': 2, 'T': 0}
Шалгалт тэнцлээ: SA=5, T=0, WA=2


## Алхам 3: Backtracking хайлт ба check_solution

In [5]:
def check_solution(csp, assignment):
    """Хайлтаас хараат бусаар: бүгд оноогдсон уу, хязгаарлалт биелсэн үү."""
    if assignment is None or any(v not in assignment for v in csp.variables):
        return False
    for v in csp.variables:
        if assignment[v] not in csp.domains[v]:
            return False
        for n in csp.neighbors[v]:
            if not csp.constraint(v, assignment[v], n, assignment[n]):
                return False
    return True

In [6]:
# ---------- Heuristic ----------
def legal_values(csp, var, assignment):
    """Одоогийн оноолттой зөрчилдөхгүй утгууд."""
    return [x for x in csp.domains[var] if csp.is_consistent(var, x, assignment)]

def select_unassigned_variable(csp, assignment, use_mrv=False, use_degree=False):
    """MRV (боломжит утга хамгийн цөөн), tie-breaker нь degree."""
    un = csp.unassigned(assignment)
    if not use_mrv:
        return un[0]
    def key(v):
        k = len(legal_values(csp, v, assignment))
        if use_degree:
            deg = sum(1 for n in csp.neighbors[v] if n not in assignment)
            return (k, -deg)
        return (k,)
    return min(un, key=key)   # тэнцвэл жагсаалтын эхнийхийг авна

def order_domain_values(csp, var, assignment, use_lcv=False):
    """LCV: оноогдоогүй хөршүүдээс хамгийн цөөн утга хасах утгыг түрүүлж."""
    vals = legal_values(csp, var, assignment)
    if not use_lcv:
        return vals
    def cost(x):
        c = 0
        for n in csp.neighbors[var]:
            if n in assignment:
                continue
            for y in legal_values(csp, n, assignment):
                if not csp.constraint(var, x, n, y):
                    c += 1
        return c
    return sorted(vals, key=cost)   # sorted нь тогтвортой

# ---------- Хайлт ----------
def backtrack(csp, assignment, use_mrv=False, use_degree=False, use_lcv=False):
    if len(assignment) == len(csp.variables):
        return dict(assignment)
    var = select_unassigned_variable(csp, assignment, use_mrv, use_degree)
    for value in order_domain_values(csp, var, assignment, use_lcv):
        if csp.is_consistent(var, value, assignment):
            csp.nodes += 1
            if csp.limit and csp.nodes > csp.limit:
                raise TooSlow()
            assignment[var] = value
            result = backtrack(csp, assignment, use_mrv, use_degree, use_lcv)
            if result is not None:
                return result
            del assignment[var]
    return None

def backtracking_search(csp, use_mrv=False, use_degree=False, use_lcv=False, limit=None):
    csp.nodes = 0
    csp.limit = limit
    return backtrack(csp, {}, use_mrv, use_degree, use_lcv)

**Шалгах:** Газрын зурагт backtracking_search-ийг ажиллуулж шийдийг хэвлэнэ.
check_solution нь True буцааж, хөрш мужууд өөр өнгөтэй байх ёстой.
Тоолуурын утгыг тэмдэглэ. Энэ нь Алхам 8-ын хүснэгтэд орно.

In [7]:
m = make_map_csp()
sol = backtracking_search(m)
print(sol)
print("zangilaa (nodes) =", m.nodes, "| check_solution =", check_solution(m, sol))
edges = [(a, b) for a in m.variables for b in m.neighbors[a] if a < b]
print("Хөрш мужуудын өнгө ялгаатай эсэх:", all(sol[a] != sol[b] for a, b in edges))

{'WA': 'red', 'NT': 'green', 'SA': 'blue', 'Q': 'red', 'NSW': 'green', 'V': 'red', 'T': 'red'}
zangilaa (nodes) = 7 | check_solution = True
Хөрш мужуудын өнгө ялгаатай эсэх: True


## Алхам 4: N-Queens

In [8]:
def make_queens_csp(n):
    """N-Queens: мөр бүр хувьсагч, домэйн нь баганы дугаар."""
    variables = list(range(n))
    domains = {v: list(range(n)) for v in variables}
    neighbors = {v: [u for u in variables if u != v] for v in variables}
    def constraint(A, a, B, b):
        return a != b and abs(a - b) != abs(A - B)
    return CSP(variables, domains, neighbors, constraint)

In [9]:
def count_solutions(csp, use_mrv=False, use_degree=False):
    """Бүх шийдийг тоолно (шийд олдсон ч зогсохгүй)."""
    csp.nodes = 0
    csp.limit = None
    cnt = 0
    def rec(assignment):
        nonlocal cnt
        if len(assignment) == len(csp.variables):
            cnt += 1
            return
        var = select_unassigned_variable(csp, assignment, use_mrv, use_degree)
        for value in csp.domains[var]:
            if csp.is_consistent(var, value, assignment):
                csp.nodes += 1
                assignment[var] = value
                rec(assignment)
                del assignment[var]
    rec({})
    return cnt

def print_board(assignment, n):
    for r in range(n):
        print(" ".join("Q" if assignment[r] == c else "." for c in range(n)))

**Шалгах:** Бүх шийдийн тоо дараах хүснэгттэй таарах ёстой.


In [10]:
expected = {1:1, 2:0, 3:0, 4:2, 5:10, 6:4, 7:40, 8:92}
for n, e in expected.items():
    got = count_solutions(make_queens_csp(n))
    print(f"N={n}: {got} (хүлээгдсэн {e})", "OK" if got == e else "АЛДАА")
    assert got == e
q4 = make_queens_csp(4)
s = backtracking_search(q4); print("\n4-Queens эхний шийд:", s); print_board(s, 4)
print("check_solution:", check_solution(q4, s))

N=1: 1 (хүлээгдсэн 1) OK
N=2: 0 (хүлээгдсэн 0) OK
N=3: 0 (хүлээгдсэн 0) OK
N=4: 2 (хүлээгдсэн 2) OK
N=5: 10 (хүлээгдсэн 10) OK
N=6: 4 (хүлээгдсэн 4) OK
N=7: 40 (хүлээгдсэн 40) OK
N=8: 92 (хүлээгдсэн 92) OK

4-Queens эхний шийд: {0: 1, 1: 3, 2: 0, 3: 2}
. Q . .
. . . Q
Q . . .
. . Q .
check_solution: True


**4-Queens-ийн хоёр шийд (гараар зурсантай харьцуулах):** баганын дараалал `(1,3,0,2)` ба `(2,0,3,1)`. Эхнийх нь хоёр дахь шийдийн толин тусгал.

In [11]:
# 4-Queens-ийн хоёр шийдийг хэвлэх
sols = []
def rec(a):
    if len(a) == 4: sols.append(dict(a)); return
    r = len(a)
    for c in range(4):
        if q4.is_consistent(r, c, a):
            a[r] = c; rec(a); del a[r]
rec({})
for s in sols:
    print([s[r] for r in range(4)]); print_board(s, 4); print()

[1, 3, 0, 2]
. Q . .
. . . Q
Q . . .
. . Q .

[2, 0, 3, 1]
. . Q .
Q . . .
. . . Q
. Q . .



## Алхам 5: MRV heuristic

**Шалгах:** Газрын зураг дээр WA = red оноосны дараа функц NT эсвэл SA-г сонгох
ёстой (тус бүр 2 боломжит утгатай).
MRV-тэй үед 8-Queens-ийн шийдийн тоо 92 хэвээр байх ёстой. Heuristic нь
шийдийн тоог өөрчилж болохгүй, зөвхөн хайлтын замыг өөрчилнө.

In [12]:
# MRV-г backtrack-д залгасан (use_mrv параметр) - дээрх select_unassigned_variable функцийг үз
m = make_map_csp()
v = select_unassigned_variable(m, {"WA": "red"}, use_mrv=True, use_degree=False)
print("WA=red-ийн дараа MRV сонгосон:", v)
assert v in ("NT", "SA")
for n in (6, 8):
    c = count_solutions(make_queens_csp(n), use_mrv=True)
    print(f"MRV-тэй {n}-Queens шийдийн тоо:", c)
assert count_solutions(make_queens_csp(8), use_mrv=True) == 92

WA=red-ийн дараа MRV сонгосон: NT
MRV-тэй 6-Queens шийдийн тоо: 4
MRV-тэй 8-Queens шийдийн тоо: 92


## Алхам 6: Degree heuristic

**Шалгах:** Газрын зураг дээр хоосон оноолттой үед бүх хувьсагчийн домэйн 3
утгатай тэнцүү тул degree шийднэ. Функц SA-г сонгох ёстой (5 хөрш).


In [13]:
m = make_map_csp()
v = select_unassigned_variable(m, {}, use_mrv=True, use_degree=True)
print("Хоосон оноолттой үед degree-ийн сонголт:", v)
assert v == "SA"

Хоосон оноолттой үед degree-ийн сонголт: SA


## Алхам 7: Least Constraining Value

**Шалгах:** Газрын зураг дээр WA = red, NT = green оноолттой үед Q хувьсагчийн
утгуудын дараалал red, blue байх ёстой. (red нь NSW-ээс 1 утга хасна, blue нь
NSW болон SA-аас нийт 2 утга хасна.)


In [14]:
m = make_map_csp()
a = {"WA": "red", "NT": "green"}
before = dict(a)
order = order_domain_values(m, "Q", a, use_lcv=True)
print("Q-ийн утгын дараалал:", order)
assert order == ["red", "blue"] and a == before   # оноолт өөрчлөгдөөгүй
print("Оноолт хэвээр:", a)

Q-ийн утгын дараалал: ['red', 'blue']
Оноолт хэвээр: {'WA': 'red', 'NT': 'green'}


## Алхам 8: Харьцуулалт

In [15]:
import time
cfgs = {"A. Heuristic-гүй": (0,0,0), "B. MRV": (1,0,0),
        "C. MRV + Degree": (1,1,0), "D. MRV + Degree + LCV": (1,1,1)}
probs = [("Газрын зураг", make_map_csp), ("8-Queens", lambda: make_queens_csp(8)),
         ("12-Queens", lambda: make_queens_csp(12)), ("16-Queens", lambda: make_queens_csp(16))]
table = {}
print(f"{'Тохиргоо':<24}" + "".join(f"{p[0]:>14}" for p in probs))
for name, (a, b, c) in cfgs.items():
    row = []
    for pname, mk in probs:
        p = mk()
        try:
            s = backtracking_search(p, a, b, c, limit=1_000_000)
            assert check_solution(p, s)
            row.append(p.nodes)
        except TooSlow:
            row.append("хэт удаан")
    table[name] = row
    print(f"{name:<24}" + "".join(f"{str(x):>14}" for x in row))

Тохиргоо                  Газрын зураг      8-Queens     12-Queens     16-Queens


A. Heuristic-гүй                     7           113           261         10052
B. MRV                               7            75           153            44
C. MRV + Degree                      7            75           153            44


D. MRV + Degree + LCV                7            39            32           332


Хязгаарлалтууд:
• Нэг багш нэг слотод хоёр хичээл заахгүй.
• Нэг бүлэг нэг слотод хоёр хичээлтэй байхгүй.
• AI хичээл DB хичээлээс өмнө орно.
• Багш B-ийн хичээлүүд зөвхөн 1, 2-р слотод орно.
Даалгавар:
1. Хичээл бүрийг хувьсагч, слотыг домэйн болгон CSP үүсгэ. Багш B-ийн
хязгаарлалтыг эхнээс нь домэйныг багасгах замаар шийд.
2. constraint функц дотор хоёр хичээл ижил багштай эсэх, ижил бүлэгтэй эсэх,
AI/DB-ийн дарааллын шаардлагыг шалга. Багш, бүлгийн мэдээллийг тусдаа
dictionary-д хадгал.
3. Алхам 8-ын хамгийн сайн тохиргоогоор шийдэж, хуваарийг слот тус бүрээр
хэвлэ.
4. Шалгалт: 1-р бүлэгт дахин нэг хичээл нэмбэл (нийт 5 хичээл, 4 слот) хайлт None
буцаах ёстой. Яагаад тэр вэ?

In [16]:
def make_timetable_csp(extra=()):
    """Цагийн хуваарь: хичээл = хувьсагч, слот = домэйн. Багш B-н хязгаарлалтыг домэйнд шингээсэн."""
    teacher = {"AI":"A", "DM":"A", "NET":"B", "OS":"B", "MATH":"C", "ENG":"C"}
    group   = {"AI":1, "DM":2, "NET":1, "OS":2, "MATH":2, "ENG":1}
    for name, (t, g) in extra:
        teacher[name] = t; group[name] = g
    V = list(teacher)
    dom = {v: ([1, 2] if teacher[v] == "B" else [1, 2, 3, 4]) for v in V}   # unary: домэйнээр
    rel = lambda A, B: teacher[A] == teacher[B] or group[A] == group[B] or {A, B} == {"AI", "DM"}
    nb = {v: [u for u in V if u != v and rel(v, u)] for v in V}
    def con(A, a, B, b):
        if teacher[A] == teacher[B] and a == b: return False   # нэг багш нэг слотод 2 хичээл
        if group[A] == group[B] and a == b: return False       # нэг бүлэг нэг слотод 2 хичээл
        if A == "AI" and B == "DM" and not a < b: return False # AI өмнө
        if A == "DM" and B == "AI" and not b < a: return False
        return True
    return CSP(V, dom, nb, con)

tt = make_timetable_csp()
s = backtracking_search(tt, use_mrv=True, use_degree=True, use_lcv=False)   # Алхам 8-ын хамгийн сайн (B/C) тохиргоо
print("nodes =", tt.nodes, "| check_solution =", check_solution(tt, s))
for slot in (1, 2, 3, 4):
    print(f"Слот {slot}:", [c for c in tt.variables if s[c] == slot])
for k in (1, 2):
    ex = [(f"X{i}", (f"T{i}", 1)) for i in range(k)]
    p = make_timetable_csp(ex)
    r = backtracking_search(p, True, True, False)
    print(f"1-р бүлэгт +{k} хичээл (нийт {len(p.variables)}): ", "None" if r is None else "шийдтэй", f"(nodes={p.nodes})")

nodes = 6 | check_solution = True
Слот 1: ['NET', 'MATH']
Слот 2: ['AI', 'OS']
Слот 3: ['DM', 'ENG']
Слот 4: []
1-р бүлэгт +1 хичээл (нийт 7):  шийдтэй (nodes=7)
1-р бүлэгт +2 хичээл (нийт 8):  None (nodes=28)


**Тайлбар:** 1-р бүлэгт 4-өөс илүү хичээл орвол (3 + 2 = 5) 4 слотод 5 хичээлийг зөрчилгүй хуваарилах боломжгүй (Дирихлегийн зарчим: нэг бүлэг нэг слотод 2 хичээлгүй). Тиймээс хайлт `None` буцаана.

## Нэмэлт даалгавар 2: Өнгийн тоо

Газрын зургийн домэйныг 2 өнгө болгож ажиллуулна. Хайлт None буцаах ёстой.
Яагаад ингэж гарсныг газрын зураг дээр заан тайлбарла. Ямар гурван муж бие
биетэйгээ хиллэдэг вэ?

In [ ]:
m2 = make_map_csp(("red", "green"))
print("2 өнгө ->", backtracking_search(m2), "| nodes =", m2.nodes)
print("Гурвалжин: WA-NT, NT-SA, WA-SA бүгд хиллэнэ ->", all(b in m2.neighbors[a] for a, b in [("WA","NT"),("NT","SA"),("WA","SA")]))

2 өнгө -> None | nodes = 4
Гурвалжин: WA-NT, NT-SA, WA-SA бүгд хиллэнэ -> True


**Тайлбар:** WA, NT, SA гурван муж бие биетэйгээ хиллэдэг (мөн NT-SA-Q, SA-Q-NSW, SA-NSW-V гурвалжнууд бий). Гурван муж хоорондоо хөрш бол хамгийн багадаа 3 өнгө хэрэгтэй тул 2 өнгө хүрэлцэхгүй, хайлт `None` буцаана. Хайлт 4 зангилаа оноосны дараа бүх салбар зөрчилд орж `None` буцсан.